In [1]:
import oceanbench

oceanbench.__version__

'0.1.4'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray
from oceanbench.core.interpolate import interpolate_1_degree

challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [
        "https://minio.dive.edito.eu/project-oceanbench/public/glonet_full_2024/20240103.zarr",
        "https://minio.dive.edito.eu/project-oceanbench/public/glonet_full_2024/20240110.zarr",
    ],
    engine="zarr",
    preprocess=lambda dataset: dataset.rename({"time": "lead_day_index"}).assign({"lead_day_index": range(10)}),
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=True,
).assign(
    {
        "first_day_datetime": [
            datetime.fromisoformat("2024-01-03"),
            datetime.fromisoformat("2024-01-10"),
        ]
    }
)
challenger_dataset = interpolate_1_degree(challenger_dataset)
challenger_dataset


<xarray.Dataset> Size: 823MB
Dimensions:                          (first_day_datetime: 2,
                                      lead_day_index: 10, depth: 21,
                                      latitude: 168, longitude: 360)
Coordinates:
  * depth                            (depth) float32 84B 0.494 ... 5.275e+03
  * lead_day_index                   (lead_day_index) int64 80B 0 1 2 ... 7 8 9
  * first_day_datetime               (first_day_datetime) datetime64[us] 16B ...
  * latitude                         (latitude) float64 1kB -77.5 -76.5 ... 89.5
  * longitude                        (longitude) float64 3kB -179.5 ... 179.5
Data variables:
    sea_water_salinity               (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 10, 1, 168, 360), meta=np.ndarray>
    sea_water_potential_temperature  (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 10, 1, 168, 360), meta=np.ndarray>
    eastward_sea_water_velocity      (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 10, 1, 168, 360), meta=np.ndarray>
    northward_sea_water_velocity     (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 10, 1, 168, 360), meta=np.ndarray>
    sea_surface_height_above_geoid   (first_day_datetime, lead_day_index, latitude, longitude) float64 10MB dask.array<chunksize=(1, 10, 168, 360), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.069441,0.075348,0.070574,0.074921,0.073913,0.080429,0.078429,0.083628,0.081878,0.089316
Temperature (°C) [sea_water_potential_temperature]{surface},0.627927,0.642316,0.674651,0.684090,0.732472,0.746026,0.802474,0.813536,0.861703,0.872216
Salinity (PSU) [sea_water_salinity]{surface},0.501832,0.500913,0.499239,0.496763,0.494219,0.493383,0.491268,0.490670,0.492273,0.491915
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.134235,0.131890,0.132082,0.132536,0.134293,0.136056,0.139860,0.142522,0.148468,0.148850
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.134836,0.133959,0.135797,0.137043,0.140335,0.140792,0.144715,0.146237,0.150498,0.152510
Temperature (°C) [sea_water_potential_temperature]{50m},1.018327,1.029979,1.055963,1.075088,1.105001,1.148240,1.186322,1.252939,1.282218,1.357534
Salinity (PSU) [sea_water_salinity]{50m},0.240008,0.240565,0.242409,0.243574,0.245910,0.248016,0.251950,0.254533,0.258722,0.260581
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.116805,0.116526,0.114696,0.115362,0.115612,0.118247,0.121759,0.126289,0.129107,0.130460
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.118878,0.118178,0.117863,0.118914,0.120296,0.122618,0.125922,0.129919,0.134014,0.138565
Temperature (°C) [sea_water_potential_temperature]{100m},1.160086,1.162242,1.169764,1.177032,1.183722,1.204335,1.218751,1.254146,1.268142,1.319533


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},44.331556,44.848067,45.446386,45.690721,47.448233,48.69253,49.592762,50.973138,51.511081,53.19573


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.059965,0.061612,0.064432,0.066465,0.069558,0.072288,0.076276,0.079799,0.083501,0.087504
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.059816,0.065590,0.071044,0.072377,0.073515,0.079164,0.085483,0.089181,0.091145,0.100556


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Temperature (°C) [sea_water_potential_temperature]{surface},0.684265,0.786599,0.774202,0.762857,0.775721,0.798761,0.850950,0.883590,0.939973,0.885074
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.553064,0.596352,0.681479,0.640969,0.659529,0.774697,0.821907,0.863910,0.832455,0.965202
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.897860,1.082088,1.042365,0.998764,1.128739,1.038721,1.078613,1.272096,1.149243,1.382733
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.929219,0.929766,0.865316,0.906925,0.856689,0.963101,1.015117,0.994653,0.949351,1.034166
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.527125,0.539896,0.518279,0.502793,0.485040,0.540035,0.605896,0.523850,0.494696,0.559908
Salinity (PSU) [sea_water_salinity]{0-5m},0.239731,0.194119,0.229352,0.268926,0.216171,0.219751,0.224384,0.231095,0.211917,0.237710
Salinity (PSU) [sea_water_salinity]{5-100m},0.189642,0.174721,0.204907,0.170564,0.190206,0.208758,0.222331,0.189485,0.190275,0.215947
Salinity (PSU) [sea_water_salinity]{100-300m},0.123570,0.130314,0.117152,0.120085,0.114049,0.144207,0.140595,0.119675,0.126293,0.139275
Salinity (PSU) [sea_water_salinity]{300-600m},0.073601,0.081194,0.075813,0.072836,0.078188,0.076801,0.087593,0.086433,0.075089,0.089228
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.109915,0.105132,0.110309,0.108286,0.113396,0.110404,0.114901,0.113708,0.118724,0.115163


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},10.80408,19.648827,27.556232,35.009098,42.14954,49.254234,56.387215,63.477333


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.020967,0.025500,0.026144,0.032648,0.039266,0.046157,0.051316,0.058375,0.060190,0.064963
Temperature (°C) [sea_water_potential_temperature]{surface},0.336703,0.367344,0.478012,0.500911,0.597998,0.620412,0.703537,0.725946,0.788026,0.804052
Salinity (PSU) [sea_water_salinity]{surface},0.108941,0.109956,0.170762,0.175415,0.218937,0.226631,0.257499,0.263784,0.287174,0.289136
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.076890,0.077889,0.078391,0.089031,0.096545,0.107417,0.118212,0.128066,0.138465,0.144102
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.073168,0.076994,0.080419,0.090449,0.100461,0.109968,0.121579,0.130816,0.138578,0.143562
Temperature (°C) [sea_water_potential_temperature]{50m},0.681733,0.682878,0.737852,0.764912,0.838635,0.890191,0.965983,1.040117,1.100159,1.176101
Salinity (PSU) [sea_water_salinity]{50m},0.089801,0.090518,0.128364,0.132159,0.154826,0.159647,0.175622,0.181413,0.193411,0.196975
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.053892,0.056348,0.061051,0.066568,0.072948,0.080996,0.090840,0.100725,0.108246,0.114128
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.051869,0.053823,0.061181,0.067769,0.075277,0.082835,0.092541,0.102811,0.112109,0.120776
Temperature (°C) [sea_water_potential_temperature]{100m},0.523697,0.531264,0.616468,0.652899,0.740299,0.795904,0.879425,0.952099,1.015811,1.091695


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.198188,41.485293,43.014469,42.933028,45.435742,46.241655,47.99596,48.831574,49.720152,50.863772


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.030204,0.03602,0.043538,0.046489,0.053446,0.058722,0.065839,0.071658,0.077109,0.081384
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.035530,0.04659,0.053187,0.055344,0.059053,0.068154,0.078017,0.084441,0.084885,0.094068


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},4.777349,8.740759,12.699625,17.154964,22.018654,27.53986,33.573021,39.971809
